# Python for AI — Class 21 · In-Class Exercise
### Neural Networks: first from scratch, then with PyTorch

Today we build a neural network **3 times**. Each time it gets a little bigger.

| Part | What we do | Tool |
|------|------------|------|
| **1** | Build one neuron and train it **by hand** | NumPy |
| **2** | Do the **same thing** with PyTorch | PyTorch |
| **3** | 💪 **Challenge:** a real network for house prices | PyTorch + sklearn |

**How to work:**
- Fill in every `___` and run the cell.
- Under each cell, the ✅ line tells you what you should see.
- Stuck? Ask your neighbour first, then the teacher.

**Using Colab?** Click the **folder icon** on the left → **upload** `students_100.csv`, `houses.csv` and `cars.csv` first.

In [ ]:
# On your own laptop, install once:  pip install torch scikit-learn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn

torch.manual_seed(42)        # the same "random" starting weights for everyone

# Part 1: From scratch (NumPy)

No new library. We build every piece ourselves, so we **see how it works inside**.

## Exercise 1: One neuron

Ali decides: **go out to play cricket?**

| Input | Value (1 = yes) | Weight |
|-------|-----------------|--------|
| Sunny? | 1 | +3 |
| Friends free? | 1 | +2 |
| Exam tomorrow? | 1 | −6 |

Bias = −1 (Ali is a little lazy).

**A neuron:** multiply each input by its weight → add them up → add the bias. Rule: total **above 0** → GO.

In [ ]:
inputs = np.array([1, 1, 1])      # sunny, friends free, exam tomorrow
weights = np.array([3, 2, -6])
bias = -1

total = ___
decision = "GO" if ___ else "STAY HOME"
print("Total:", total, "→", decision)

✅ You should see: `Total: -2 → STAY HOME`

**Try it:** change `inputs` to `[1, 1, 0]` (no exam). What does Ali decide now?

## Exercise 2: Activations

- **ReLU:** negative → 0. Otherwise keep the number. Used **inside** the network.
- **Sigmoid:** squeezes any number into **0 to 1** (a chance). We give you this one.

Hint: `np.maximum(0, total)` keeps the bigger of 0 and the number.

In [ ]:
def relu(total):
    return ___

def sigmoid(total):
    return 1 / (1 + np.exp(-total))

print("ReLU:   ", relu(np.array([-3, 0, 2.5])))
print("Sigmoid:", sigmoid(np.array([-5, 0, 5])).round(2))

✅ You should see: ReLU `[0. 0. 2.5]` and Sigmoid `[0.01 0.5 0.99]`

## Exercise 3: A forward pass through a small network

A student studies **3 hours** and uses the phone **2 hours**. The network guesses the Maths mark.

```
 study_hours ──┬──▶ (neuron 1) ──┐
               ├──▶ (neuron 2) ──┼──▶ (output) ──▶ mark
 phone_hours ──┴──▶ (neuron 3) ──┘
```

- **Hidden layer:** `x @ W1 + b1`, then ReLU. (`@` = multiply and add, for all 3 neurons at once.)
- **Output:** `hidden @ W2 + b2`

The weights are **random** for now. The network has not learned anything yet.

In [ ]:
x_student = np.array([3, 2])                   # study_hours, phone_hours

W1 = np.array([[ 0.5, -1.0, 0.8],              # from study_hours to neurons 1, 2, 3
               [-0.6,  0.4, 0.2]])             # from phone_hours to neurons 1, 2, 3
b1 = np.array([0.1, 0.2, -0.5])
W2 = np.array([2, 1, 3])                       # from the 3 hidden neurons to the output
b2 = 1

hidden = relu(___)
output = ___
print("hidden:", hidden.round(2))
print("output:", round(output, 2))

✅ You should see: `hidden: [0.4 0. 2.3]` and `output: 8.7`

8.7 is a **bad** guess for a Maths mark (real: about 55). The weights are random. **Learning = finding better weights.**

## Exercise 4: Loss: how wrong is a line?

The simplest network is **one neuron**: `prediction = w × study_hours + b`.

**Loss (MSE):** take each error (prediction − real mark), square it, then take the average. Smaller = better.

In [ ]:
students = pd.read_csv("students_100.csv")
x = students["study_hours"].values
y = students["maths"].values

def loss(w, b):
    prediction = ___
    return ___

print("bad line  (w=0, b=50):", round(loss(0, 50)))
print("good line (w=9, b=28):", round(loss(9, 28)))

✅ You should see: the good line has a **much smaller** loss (about 177 vs 71).

## Exercise 5: One step downhill

**Gradient descent:** like walking down a hill in the fog. Feel which way goes down, take a **small step**.

- `grad_w` and `grad_b` tell us **which way is uphill**. We give you the formulas (no calculus today!).
- So we step the **other way**: `new w = w − learning_rate × grad_w`

In [ ]:
w, b = 0.0, 0.0               # start with a bad guess
learning_rate = 0.05

error = (w * x + b) - y
grad_w = (2 * error * x).mean()     # which way is uphill for w (given)
grad_b = (2 * error).mean()         # which way is uphill for b (given)

print("loss before:", round(loss(w, b)))
w = ___
b = ___
print("loss after: ", round(loss(w, b)))

✅ You should see: `loss after` is **much smaller** than `loss before`. One step, and we are already better!

## Exercise 6: Repeat the step = training!

Do Exercise 5 again and again. **One round through all the data = one epoch.**

In [ ]:
def train(x, y, learning_rate, epochs):
    w, b = 0.0, 0.0
    losses = []
    for epoch in range(epochs):
        error = (w * x + b) - y                  # 1. forward pass + error
        losses.append((error ** 2).mean())       # 2. loss
        w = ___   # 3. step downhill
        b = ___
    return w, b, losses

w, b, losses = train(x, y, learning_rate=0.05, epochs=2000)
print("Our neuron: w =", round(w, 2), "  b =", round(b, 2))
print("np.polyfit:", np.polyfit(x, y, 1).round(2))

plt.plot(losses)
plt.ylim(0, 500)
plt.title("Loss during training")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

✅ You should see: `w ≈ 9.19` and `b ≈ 28.09`, the **same line** as `np.polyfit`. The loss curve goes down, then becomes flat.

Our neuron found the best line **by itself**, only by taking small steps downhill.

## Exercise 7: Play with the learning rate

No blanks here. Run it and **answer the question** below.

In [ ]:
for lr in [0.001, 0.05, 0.12]:
    w, b, losses = train(x, y, learning_rate=lr, epochs=50)
    plt.plot(losses, label=f"learning rate {lr}")

plt.yscale("log")        # the loss can get huge, so we use a log scale
plt.xlabel("Epoch")
plt.ylabel("Loss (log scale)")
plt.legend()
plt.show()

**Question:** which learning rate is too small? Which one is just right? Which one **explodes**? Why?

*Your answer:* ...

### ✔️ Part 1 done! What you built

| Piece | Your code |
|-------|-----------|
| Neuron | `inputs × weights + bias` |
| Forward pass | `relu(x @ W1 + b1) @ W2 + b2` |
| Loss | `((prediction - y) ** 2).mean()` |
| Gradient descent | `w = w - learning_rate * grad_w` |
| Training | repeat the step for many **epochs** |

**The problem:** we wrote the gradient formulas by hand. A real network has **millions** of weights. Nobody can write all those formulas. That is why we use **PyTorch**.

# Part 2: The same thing, with PyTorch

PyTorch is the library real AI companies use. It finds the gradients **for us** (backpropagation), with one line.

| By hand (Part 1) | PyTorch |
|------------------|---------|
| `np.array` | `torch.tensor` |
| `w * x + b` | `nn.Linear(1, 1)` |
| `((prediction - y) ** 2).mean()` | `nn.MSELoss()` |
| write `grad_w` by hand | `loss.backward()` |
| `w = w - learning_rate * grad_w` | `optimizer.step()` |

## Exercise 8: Tensors

A **tensor** = PyTorch's array. Like NumPy, but it can find gradients.

Networks want data like a table: one **row** per student, one **column** per feature. `.reshape(-1, 1)` makes one column.

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
print(a * 2, a.mean())

x_t = torch.tensor(x, dtype=torch.float32).reshape(-1, 1)
y_t = ___
print(x_t.shape, y_t.shape)

✅ You should see: `torch.Size([100, 1]) torch.Size([100, 1])`

## Exercise 9: PyTorch finds the gradient

A tiny problem: input 3, we want the output 12. So the best `w` is 4. We start at `w = 2`.

- `requires_grad=True` = "please track this number".
- `.backward()` = **backpropagation**: find the gradient. The answer goes in `w.grad`.

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
loss_value = (w * 3 - 12) ** 2

___
print("gradient of w:", w.grad.item())

✅ You should see: `-36.0`. **Negative** → make `w` **bigger**. Correct: 2 is too small, the best is 4.

We wrote **no formula**. Compare with Exercise 5!

## Exercise 10: The training loop in 5 steps

Every PyTorch training, small or huge, has the **same 5 lines**:

1. `prediction = model(x)` → guess
2. `loss = loss_fn(prediction, y)` → how wrong
3. `optimizer.zero_grad()` → clear the old gradients
4. `loss.backward()` → find the new gradients
5. `optimizer.step()` → step downhill (change all weights)

In [ ]:
torch.manual_seed(42)
neuron = nn.Linear(1, 1)                                   # one neuron: w * x + b
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(neuron.parameters(), lr=0.05)

for epoch in range(2000):
    prediction = ___
    loss = ___
    optimizer.zero_grad()
    ___
    ___

print("w =", round(neuron.weight.item(), 2), "  b =", round(neuron.bias.item(), 2))

✅ You should see: `w ≈ 9.19` and `b ≈ 28.09`

**Three ways, one answer:** `np.polyfit`, our hand-made gradient descent, and PyTorch.

## Exercise 11: A bigger network

`nn.Sequential` puts layers **in a row**:

```
 4 inputs  ──▶  8 hidden neurons (ReLU)  ──▶  1 output
```

**Guess first:** how many weights and biases? (Layer 1: 4 × 8 weights + 8 biases. Layer 2: 8 × 1 weights + 1 bias.)

In [ ]:
model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
)
print(model)
print("Weights and biases:", sum(p.numel() for p in model.parameters()))

✅ You should see: `49` (32 + 8 + 8 + 1). They all start **random**. Training will fix them.

# Part 3: 💪 Challenge: house prices with a real network

**The problem:** a property dealer in Lahore asks:
> "Everyone talks about neural networks. Is a neural network better than Linear Regression for my house prices?"

This is a **real** workflow. Two new things:

| New thing | Why |
|-----------|-----|
| **Train / validation / test** (60 / 20 / 20) | train = homework, validation = **mock exam** during training, test = **final exam**, only once at the end |
| **StandardScaler** | networks learn much better when all inputs have the same size (like KNN in Class 19) |
| **Adam** | a smarter optimizer than SGD: it changes the step size by itself |
| `with torch.no_grad():` | "only guessing, no learning here" |

### Step 1: Load the data

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

houses = pd.read_csv("houses.csv")
X = houses[["size_marla", "bedrooms", "age_years", "distance_km"]].values
y = houses["price_lakh"].values
print(X.shape, y.shape)

### Step 2: Split into train / validation / test

We split **two times**: first take out the test part (20%), then split the rest into train and validation.

In [ ]:
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_rest, y_rest, test_size=___, random_state=42)   # 25% of 80% = 20%

print("train:", len(X_train), " validation:", len(X_val), " test:", len(X_test))

✅ You should see: about 60% / 20% / 20% of the rows.

### Step 3: Scale, then make tensors

Fit the scaler on **train only**. (The test is the "final exam": we don't peek at it.)

In [ ]:
scaler = StandardScaler().fit(___)

def to_tensor(a):
    return torch.tensor(a, dtype=torch.float32)

X_train_t = to_tensor(scaler.transform(X_train))
X_val_t   = to_tensor(scaler.transform(X_val))
X_test_t  = to_tensor(scaler.transform(X_test))
y_train_t = to_tensor(y_train).reshape(-1, 1)
y_val_t   = to_tensor(y_val).reshape(-1, 1)

### Step 4: Build the network, the loss and the optimizer

In [ ]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(___, 8),          # how many inputs? (count the features!)
    nn.ReLU(),
    nn.Linear(8, 1),
)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)

### Step 5: Train, and save both losses after every epoch

Write the **5 steps** yourself. You know them from Exercise 10!

In [ ]:
train_losses, val_losses = [], []

for epoch in range(500):
    # learn on the train data (the 5 steps)
    prediction = ___
    loss = ___
    ___
    ___
    ___
    train_losses.append(loss.item())

    # check on the validation data (no learning here)
    with torch.no_grad():
        val_losses.append(loss_fn(model(X_val_t), y_val_t).item())

print("Last train loss:     ", round(train_losses[-1]))
print("Last validation loss:", round(val_losses[-1]))

### Step 6: Read the loss curves

| Shape | Meaning |
|-------|---------|
| both go down and stay **close** | ✅ good |
| validation goes **up** again, train keeps going down | ⚠️ overfitting (memorising) |
| lines **jump** up and down | ⚠️ learning rate too big |

In [ ]:
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="validation")
plt.ylim(0, 2000)                  # the first epochs are huge; zoom in on the rest
plt.title("Loss curves")
plt.xlabel("Epoch")
plt.ylabel("Loss (MSE)")
plt.legend()
plt.show()

**Question:** which shape do you see? Good, overfitting, or learning rate too big?

### Step 7: The final exam: test data. Compare with Linear Regression

**MAE** = on average, how many lakh is the guess wrong?

In [ ]:
with torch.no_grad():
    nn_pred = model(X_test_t).numpy().ravel()          # .ravel() = one flat list
nn_mae = mean_absolute_error(y_test, nn_pred)

lin = LinearRegression().fit(X_train, y_train)          # Class 18's model, same train data
lin_mae = mean_absolute_error(y_test, ___)

print("Neural network MAE:   ", round(nn_mae, 1), "lakh")
print("Linear Regression MAE:", round(lin_mae, 1), "lakh")

✅ You should see: both are **about the same** (around 12.2 vs 11.9 lakh).

**Our answer to the dealer:**
> "No, a neural network is not better here. House prices in this data follow a simple straight-line pattern, and Linear Regression already finds it. It is also faster and easier to explain. We choose Linear Regression."

**The big lesson:** neural networks are not magic. They win on **complex** data: photos, sound, text. For a small, simple table, **always try a simple model first**.

## The complete program

Everything from Part 3 in **one cell**. This is what a real neural network project looks like.

In [ ]:
import pandas as pd
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# 1. data
houses = pd.read_csv("houses.csv")
X = houses[["size_marla", "bedrooms", "age_years", "distance_km"]].values
y = houses["price_lakh"].values

# 2. split 60 / 20 / 20
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_rest, y_rest, test_size=0.25, random_state=42)

# 3. scale + tensors
scaler = StandardScaler().fit(X_train)
def to_tensor(a):
    return torch.tensor(a, dtype=torch.float32)
X_train_t, X_val_t, X_test_t = (to_tensor(scaler.transform(d)) for d in (X_train, X_val, X_test))
y_train_t, y_val_t = to_tensor(y_train).reshape(-1, 1), to_tensor(y_val).reshape(-1, 1)

# 4. network, loss, optimizer
torch.manual_seed(42)
model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)

# 5. train
for epoch in range(500):
    loss = loss_fn(model(X_train_t), y_train_t)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# 6. validation check
with torch.no_grad():
    print("Validation loss:", round(loss_fn(model(X_val_t), y_val_t).item()))

# 7. final exam + compare
with torch.no_grad():
    nn_mae = mean_absolute_error(y_test, model(X_test_t).numpy().ravel())
lin_mae = mean_absolute_error(y_test, LinearRegression().fit(X_train, y_train).predict(X_test))
print("Neural network MAE:   ", round(nn_mae, 1), "lakh")
print("Linear Regression MAE:", round(lin_mae, 1), "lakh")

# 🏠 Home Problem: car prices with a neural network

The used-car showroom asks the **same question**: *"Is a neural network better?"*

Use `cars.csv`. Features: `car_age`, `km_thousands`, `engine_cc`. Target: `price_lakh`. **Do the same 7 steps as Part 3:**

1. Load the data. Make `X` (3 features) and `y`.
2. Split 60 / 20 / 20 with `random_state=42`.
3. Scale (fit on train). Make tensors.
4. Build the network. **Careful: 3 inputs this time!** Use `Adam` with `lr=0.05`.
5. Train for 500 epochs. Save train and validation losses.
6. Draw the loss curves. Which shape is it?
7. Compare test MAE with Linear Regression. What do you tell the showroom?

In [ ]:
# Write your code here

# Key Takeaways

- **Neuron:** inputs × weights + bias → activation (**ReLU** inside, **sigmoid** for a chance).
- **Forward pass:** data goes from the inputs, through the layers, to the answer.
- **Loss** = how wrong. **Learning = making the loss smaller.**
- **Gradient descent:** small steps downhill. **Learning rate** = the step size.
- **PyTorch** does the gradients for us: `loss.backward()` (backpropagation).
- The training loop: **predict → loss → zero_grad → backward → step**.
- **Train / validation / test** = homework / mock exam / final exam.
- Neural networks are not magic: **compare with a simple model first.**